In [ ]:
import os
import math
import numpy as np
import pandas as pd

In [ ]:
# ===============================
# Parameters for solar and storage
# ===============================
sc = 'column30'  # column name for solar collector potential
heat = 'column3'  # column name for heating demand

store = {
    'min': 500,            # minimum TES volume in liters
    'residential': 1500,   # max TES volume for residential in liters
    'non_residential': 3000,  # max TES volume for non-residential in liters
    'cap': 70,             # TES volume-to-capacity ratio in kWh/m^3
    'q_l': 0.99            # TES efficiency, hour-to-hour
}

min_sc_area = 10           # minimum installed solar collector area
sc_module_area = 2.5       # solar collector module area
min_sc_num = math.floor(min_sc_area / sc_module_area)



In [7]:
# ===============================
# Cost and gas functions
# ===============================
def cost(area):
    """Calculate CAPEX and OPEX based on installed area"""
    dic = {'cost': 0, 'ins_cost': 0, 'capex': 0, 'opex': 0}
    if area != 0:
        dic['cost'] = 400 * area + 2500
        dic['ins_cost'] = 0.25 * dic['cost']
        dic['capex'] = dic['cost'] + dic['ins_cost']
        dic['opex'] = 3.5 * area + 147.5
    return dic


def vol(area, usage):
    """Calculate storage volume based on area and building usage"""
    v = 40 * area + 220
    return max(store['min'], min(v, store[usage]))


def store_cap(vol):
    """Convert storage volume to storage capacity (kWh)"""
    return vol * store['cap'] / 1000


def gas(h):
    """Calculate gas cost based on demand h (kWh)"""
    h = h / 0.8
    if h <= 2000:
        return 6.00 * 12 + (20.47 + 2.226) * h / 100
    elif h <= 10000:
        return 8.00 * 12 + (18.62 + 2.226) * h / 100
    elif h <= 30000:
        return 12.00 * 12 + (17.73 + 2.226) * h / 100
    elif h <= 100000:
        return 14.00 * 12 + (17.61 + 2.226) * h / 100
    elif h <= 300000:
        return 24.00 * 12 + (17.37 + 2.226) * h / 100
    else:
        return 60.00 * 12 + (17.2 + 2.226) * h / 100


# ===============================
# NPV functions
# ===============================
def initial_constants(dic, area):
    """Initialize CAPEX, OPEX, and gas savings"""
    dic['gas_savings'] = dic['gas_cost'] - dic['remain_gas_cost']
    result = cost(area)
    dic['capex'] = result['capex']
    dic['opex'] = result['opex']
    return dic


def npv_cal(dic, i):
    """Calculate discounted cash flow for year i"""
    gas_increase = 0.0449     # annual gas price increase
    interest_rate = 0.0337    # discount rate
    c = dic['gas_savings'] * (1 + gas_increase) ** i - dic['opex']
    cur = c / (1 + interest_rate) ** i
    return cur


def npv(dic, years):
    """Calculate total NPV over given years"""
    dic['n_0'] = -1 * dic['capex']
    dic['npv'] = dic['n_0']
    for i in range(1, years + 1):
        dic['n_' + str(i)] = npv_cal(dic, i)
        dic['npv'] += dic['n_' + str(i)]
    return dic


# ===============================
# Storage simulation
# ===============================
def cal_battery(o_df, total_cap, scale):
    """Simulate hourly storage operation"""
    q_l = store['q_l']
    df = o_df.copy()

    scaled_sc = 'scaled_sc'
    o = 'over_supply'
    us = 'under_supply'
    need = 'need'
    battery = 'battery'
    after_battery = 'heat_aft_battery'

    df[scaled_sc] = df[sc] * scale
    df[o] = np.where(df[scaled_sc] <= df[heat], 0, df[scaled_sc] - df[heat])
    df[us] = np.where(df[o] == 0, df[scaled_sc], df[heat])
    df[battery] = 0
    df[need] = df[heat] - df[us]
    df[after_battery] = df[need]

    df.at[0, battery] = df.at[0, scaled_sc] - df.at[0, need]
    if df.at[0, battery] < 0:
        df.at[0, battery] = 0

    for i in range(1, len(df)):
        cur = df.at[i - 1, battery] * q_l
        if df.at[i, o] == 0 and df.at[i, heat] > 0 and cur > 0:
            need_val = df.at[i, need]
            withdraw = min(need_val, cur)
            cur -= withdraw
            df.at[i, after_battery] = need_val - withdraw
        cur += df.at[i, o]
        cur = min(cur, total_cap)
        df.at[i, battery] = cur

    return df, df[after_battery].sum(), df[heat].sum() - df[after_battery].sum()


# ===============================
# System installation optimization
# ===============================
def install_systems(df, max_area, usage):
    """Optimize solar + storage system configuration to maximize NPV"""
    max_num = int(max_area // sc_module_area)
    original_num = max_area / sc_module_area

    best_dic = {'heat': df[heat].sum()}
    best_dic['gas_cost'] = gas(best_dic['heat'])
    best_dic['need_aft_battery'] = best_dic['heat']
    best_dic['remain_gas_cost'] = best_dic['gas_cost']
    best_dic['num_sc'] = 0
    best_dic['total_installed_area'] = 0
    best_dic['total_storage'] = 0
    best_dic['total_capacity'] = 0
    best_dic['saved_aft_battery'] = 0
    best_dic = initial_constants(best_dic, 0)
    best_dic = npv(best_dic, 25)

    return_df = df.copy()
    eff = df[sc].sum() / max_area

    min_area = 6812.5 / (5.675 * eff - 587.5)
    if max_num < min_sc_num or eff < 587.5 / 5.675 or min_area > max_area:
        return best_dic, df

    min_num = max(min_sc_num, math.floor(min_area / sc_module_area))

    for i in range(max_num, min_num - 1, -1):
        total_area = i * sc_module_area
        battery_vol = vol(total_area, usage)
        total_cap = store_cap(battery_vol)
        temp_df, after_battery, saved = cal_battery(df, i, i / original_num)

        temp_dic = best_dic.copy()
        temp_dic['need_aft_battery'] = after_battery
        temp_dic['remain_gas_cost'] = gas(temp_dic['need_aft_battery'])
        temp_dic['num_sc'] = i
        temp_dic['total_installed_area'] = total_area
        temp_dic['total_storage'] = battery_vol
        temp_dic['total_capacity'] = total_cap
        temp_dic = initial_constants(temp_dic, total_area)
        temp_dic['saved_aft_battery'] = saved
        temp_dic = npv(temp_dic, 25)

        if temp_dic['npv'] >= best_dic['npv']:
            best_dic = temp_dic.copy()
            return_df = temp_df.copy()

    return best_dic, return_df



In [8]:
# ===============================
# Main pipeline adapted to new data structure
# ===============================
def run_pipeline():
    # File paths
    data_dir = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent"
    db_table_file = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\dbTable.csv"
    ts_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_ts_kl.csv"
    building_use_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_kl_buildinguse.csv"
    building_area_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_kl.csv"

    # Load reference data
    db_table = pd.read_csv(db_table_file)
    ts_map = pd.read_csv(ts_file)
    building_use_df = pd.read_csv(building_use_file)
    building_area_df = pd.read_csv(building_area_file)

    # Summary dictionary
    summary = {
        'filename': [], 'building': [], 'buildinguse': [], 'roof_area': [],
        'num_sc': [], 'total_installed_area': [], 'total_capacity': [],
        'capex': [], 'opex': [], 'need_aft_battery': [], 'saved_aft_battery': [],
        'npv': []
    }
    for j in range(0, 26):
        summary['n_' + str(j)] = []

    # Iterate over all timeseries files
    for file in os.listdir(data_dir):
        if file.endswith(".csv"):
            file_uuid = file.replace(".csv", "")
            timeseries_id = "Timeseries_" + file_uuid

            ts_row = db_table[db_table["timeseriesIRI"].str.contains(timeseries_id)]
            if ts_row.empty:
                continue
            timeseriesIRI = ts_row.iloc[0]["timeseriesIRI"]

            ts_row2 = ts_map[ts_map["Measurement"] == timeseriesIRI]
            if ts_row2.empty:
                continue
            building = ts_row2.iloc[0]["building"]

            bu_row = building_use_df[building_use_df["building"] == building]
            buildinguse = bu_row.iloc[0]["buildinguse"] if not bu_row.empty else "non_residential"
            usage = "residential" if "Residential" in buildinguse or "Domestic" in buildinguse else "non_residential"
            print(f"Processing file: {file}, Building: {building}, Usage: {usage}")
            area_row = building_area_df[
                (building_area_df["building"] == building) &
                (building_area_df["Property"] == "Roof solar suitable area")
            ]
            if area_row.empty:
                continue
            roof_area = float(area_row.iloc[0]["Value"])

            df = pd.read_csv(os.path.join(data_dir, file))
            result, _ = install_systems(df, roof_area, usage)

            # Append results
            summary['filename'].append(file)
            summary['building'].append(building)
            summary['buildinguse'].append(buildinguse)
            summary['roof_area'].append(roof_area)
            summary['num_sc'].append(result['num_sc'])
            summary['total_installed_area'].append(result['total_installed_area'])
            summary['total_capacity'].append(result['total_capacity'])
            summary['capex'].append(result['capex'])
            summary['opex'].append(result['opex'])
            summary['need_aft_battery'].append(result['need_aft_battery'])
            summary['saved_aft_battery'].append(result['saved_aft_battery'])
            summary['npv'].append(result['npv'])
            for j in range(0, 26):
                summary['n_' + str(j)].append(result['n_' + str(j)])

    # Save results
    out_file = "npv_withstorage_results.csv"
    pd.DataFrame(summary).to_csv(out_file, index=False)
    print(f"✅ Results saved to {out_file}")


In [18]:
def run_pipeline(debug=True, batch_size=10):
    # File paths
    data_dir = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent"
    db_table_file = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\dbTable.csv"
    ts_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_ts_kl.csv"
    building_use_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_kl_buildinguse.csv"
    building_area_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_kl.csv"

    # Load reference CSVs
    db_table = pd.read_csv(db_table_file)
    ts_map = pd.read_csv(ts_file)
    building_use_df = pd.read_csv(building_use_file)
    building_area_df = pd.read_csv(building_area_file)

    out_file = "npv_results.csv"
    header_written = False
    batch_results = []

    # Loop through all timeseries CSV files
    for idx, file in enumerate(os.listdir(data_dir), start=1):
        if not file.endswith(".csv"):
            continue

        try:
            file_uuid = file.replace(".csv", "")
            if debug:
                print(f"\n🔎 Processing file: {file}")

            # Step 1: match in dbTable.csv
            ts_row = db_table[db_table["tableName"].str.contains(file_uuid, na=False)]
            if ts_row.empty:
                print(f"  ❌ No match in dbTable.csv for {file_uuid}")
                continue
            dataIRI = ts_row.iloc[0]["dataIRI"]

            # Step 2: match building in results_ts_kl.csv
            ts_row2 = ts_map[ts_map["Measurement"] == dataIRI]
            if ts_row2.empty:
                print(f"  ❌ No building found for {dataIRI}")
                continue
            building = ts_row2.iloc[0]["building"]

            # Step 3: buildinguse
            bu_row = building_use_df[building_use_df["building"] == building]
            buildinguse = bu_row.iloc[0]["buildinguse"] if not bu_row.empty else "non_residential"
            usage = "residential" if "Residential" in buildinguse or "Domestic" in buildinguse else "non_residential"

            # Step 4: roof area
            area_row = building_area_df[
                (building_area_df["building"] == building) &
                (building_area_df["Property"] == "Roof solar suitable area")
            ]
            if area_row.empty:
                print(f"  ❌ No roof area for {building}")
                continue
            roof_area = float(area_row.iloc[0]["Value"])

            # Step 5: load timeseries
            df = pd.read_csv(os.path.join(data_dir, file))
            if not {"column3", "column30"}.issubset(df.columns):
                print(f"  ❌ Missing required columns in {file}")
                continue

            # Step 6: run NPV calculation
            result, _ = install_systems(df, roof_area, usage)

            # Build row dict
            row = {
                'filename': file,
                'building': building,
                'buildinguse': buildinguse,
                'roof_area': roof_area,
                'num_sc': result['num_sc'],
                'total_installed_area': result['total_installed_area'],
                'total_capacity': result['total_capacity'],
                'capex': result['capex'],
                'opex': result['opex'],
                'need_aft_battery': result['need_aft_battery'],
                'saved_aft_battery': result['saved_aft_battery'],
                'npv': result['npv']
            }
            for j in range(0, 26):
                row[f'cashflow_year{j}'] = result['n_' + str(j)]

            batch_results.append(row)

            # Write batch to CSV every batch_size files
            if len(batch_results) >= batch_size:
                pd.DataFrame(batch_results).to_csv(
                    out_file, mode="a", header=not header_written, index=False
                )
                header_written = True
                batch_results = []  # clear buffer
                if debug:
                    print(f"  💾 Saved batch up to file #{idx}")

        except Exception as e:
            print(f"  ❌ Error processing {file}: {e}")

    # Save remaining results (if any)
    if batch_results:
        pd.DataFrame(batch_results).to_csv(
            out_file, mode="a", header=not header_written, index=False
        )
        if debug:
            print(f"  💾 Saved final batch with {len(batch_results)} rows")

    print(f"\n✅ All results saved to {out_file}")

In [ ]:
if __name__ == "__main__":
    run_pipeline()

In [ ]:
def run_pipeline_PS(debug=True, batch_size=10):
    # File paths (PS city)
    data_dir = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\PS_TS\CEAAgent"
    db_table_file = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\PS_TS\dbTable.csv"
    ts_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_ts_ps.csv"
    building_use_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_ps_buildinguse.csv"
    building_area_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_ps.csv"

    # Load reference CSVs
    db_table = pd.read_csv(db_table_file)
    ts_map = pd.read_csv(ts_file)
    building_use_df = pd.read_csv(building_use_file)
    building_area_df = pd.read_csv(building_area_file)

    out_file = "npv_ps_results.csv"
    header_written = False
    batch_results = []

    # Loop through all timeseries CSV files
    for idx, file in enumerate(os.listdir(data_dir), start=1):
        if not file.endswith(".csv"):
            continue

        try:
            file_uuid = file.replace(".csv", "")
            if debug:
                print(f"\n🔎 Processing file: {file} (uuid={file_uuid})")

            # Step 1: match in dbTable.csv
            ts_row = db_table[db_table["tableName"].str.contains(file_uuid, na=False)]
            if ts_row.empty:
                print(f"  ❌ No match in dbTable.csv for {file_uuid}")
                continue
            dataIRI = ts_row.iloc[0]["dataIRI"]

            # Step 2: match building in results_ts_ps.csv
            ts_row2 = ts_map[ts_map["Measurement"] == dataIRI]
            if ts_row2.empty:
                print(f"  ❌ No building found in results_ts_ps.csv for {dataIRI}")
                continue
            building = ts_row2.iloc[0]["building"]

            # Step 3: buildinguse
            bu_row = building_use_df[building_use_df["building"] == building]
            buildinguse = bu_row.iloc[0]["buildinguse"] if not bu_row.empty else "non_residential"
            usage = "residential" if "Residential" in buildinguse or "Domestic" in buildinguse else "non_residential"

            # Step 4: roof area
            area_row = building_area_df[
                (building_area_df["building"] == building) &
                (building_area_df["Property"] == "Roof solar suitable area")
            ]
            if area_row.empty:
                print(f"  ❌ No roof area found in results_ps.csv for building {building}")
                continue
            roof_area = float(area_row.iloc[0]["Value"])

            # Step 5: load timeseries CSV
            df = pd.read_csv(os.path.join(data_dir, file))
            if not {"column3", "column30"}.issubset(df.columns):
                print(f"  ❌ Missing required columns in {file}")
                continue

            # Step 6: run NPV calculation (with storage)
            result, _ = install_systems(df, roof_area, usage)
            if debug:
                print(f"  ✅ NPV calculated: {result['npv']:.2f}")

            # Build row dict
            row = {
                'filename': file,
                'building': building,
                'buildinguse': buildinguse,
                'roof_area': roof_area,
                'num_sc': result['num_sc'],
                'total_installed_area': result['total_installed_area'],
                'total_capacity': result['total_capacity'],
                'capex': result['capex'],
                'opex': result['opex'],
                'need_aft_battery': result['need_aft_battery'],
                'saved_aft_battery': result['saved_aft_battery'],
                'npv': result['npv']
            }
            for j in range(0, 26):
                row[f'cashflow_year{j}'] = result['n_' + str(j)]

            batch_results.append(row)

            # Write batch to CSV every batch_size files
            if len(batch_results) >= batch_size:
                pd.DataFrame(batch_results).to_csv(
                    out_file, mode="a", header=not header_written, index=False
                )
                header_written = True
                batch_results = []  # clear buffer
                if debug:
                    print(f"  💾 Saved batch up to file #{idx}")

        except Exception as e:
            print(f"  ❌ Error processing {file}: {e}")

    # Save remaining results
    if batch_results:
        pd.DataFrame(batch_results).to_csv(
            out_file, mode="a", header=not header_written, index=False
        )
        if debug:
            print(f"  💾 Saved final batch with {len(batch_results)} rows")

    print(f"\n✅ All results saved to {out_file}")


In [ ]:
if __name__ == "__main__":
    run_pipeline_PS()